# 🚨 Step 4 — Score Transactions & Generate Fraud Alerts (Gold)

Apply the trained model to all transactions.
Flag suspicious ones with a fraud probability score.

In [ ]:
# Import model loading utilities and pandas helpers used during fraud scoring
import mlflow.sklearn
import pandas as pd

# Define the MLflow model URI pattern for loading a previously trained fraud model
model_uri = 'runs:/{run_id}/fraud_model'  # Replace run_id from previous notebook output
# For demo: reload model from training session
# model = mlflow.sklearn.load_model(model_uri)

# Re-train the model inline so the scoring demo can run end-to-end without a manual run_id
from sklearn.ensemble import GradientBoostingClassifier
df_all = spark.table('silver_fraud_features').toPandas()

# Reuse the same engineered features from training when fitting the scoring model
features = ['Amount','TimeSinceLastTxnMins','NumTxnLast24h','AvgTxnAmount30d',
            'location_jump','high_velocity','amount_spike','fast_repeat']

# Fit the fraud model on all available engineered transactions before scoring
model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(df_all[features], df_all['IsFraud'].astype(int))

# Confirm the model is ready to score incoming transactions
print('✅ Model ready for scoring')

In [ ]:
# Copy the engineered dataset so fraud scores can be added without changing the original data
df_score = df_all.copy()

# Calculate fraud probability for each transaction and flag high-risk records above the threshold
df_score['FraudProbability'] = model.predict_proba(df_all[features])[:, 1]
df_score['FraudAlert'] = df_score['FraudProbability'] > 0.5

# Filter suspicious transactions and sort them from highest to lowest fraud risk
alerts = df_score[df_score['FraudAlert']][['TransactionID','AccountID','Amount','Location','FraudProbability']]
alerts = alerts.sort_values('FraudProbability', ascending=False)

# Display the alert count and detailed high-risk transactions for analyst review
print(f'🚨 FRAUD ALERTS: {len(alerts)} suspicious transactions detected!')
print()
print(alerts.to_string(index=False))

In [ ]:
# Select the scored transaction fields that will be published to the Gold fraud alerts table
df_gold = spark.createDataFrame(df_score[['TransactionID','AccountID','Amount',
    'TransactionDate','Location','FraudProbability','FraudAlert','IsFraud']])

# Save the scored fraud alerts to the Gold Delta table for dashboards and reporting
df_gold.write.format('delta').mode('overwrite').saveAsTable('gold_fraud_alerts')

# Confirm the Gold fraud alerts table is ready for downstream consumers
print('\n✅ Gold fraud alerts table saved — ready for Power BI dashboard!')

In [ ]:
%%sql
-- Query the Gold alerts table and show the transactions with the highest fraud risk first
-- Final view: All high-risk fraud alerts
SELECT TransactionID, AccountID, Amount, Location,
       ROUND(FraudProbability * 100, 1) AS FraudScore_Pct,
       CASE WHEN FraudAlert = true THEN '🚨 ALERT' ELSE '✅ OK' END AS Status
FROM gold_fraud_alerts
ORDER BY FraudProbability DESC